# adaptive-finance-llm — Colab quickstart

Mounts Google Drive, clones the repo (no local Mac environment needed —
everything below runs inside this Colab session), installs dependencies,
runs a fast smoke test, and (optionally, separately) runs the baseline
training config on a Tesla T4. All data, checkpoints, and logs are written
to Drive, not the ephemeral Colab disk.

**Run the smoke test cell before the training cell.** It validates model
init, forward/backward, and checkpoint save/reload in seconds, without
downloading any dataset or running real training — catch a broken
environment early instead of after a long run.

In [ ]:
# Which branch to check out. Update to 'main' once the feature branch is
# merged; until then this points at the branch under review.
REPO_BRANCH = 'feature/initial-scaffold'
REPO_URL = 'https://github.com/ronith012/adaptive-finance-llm.git'
REPO_DIR = 'adaptive-finance-llm'
REPO_PATH = '/content/' + REPO_DIR

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.ismount('/content/drive'), (
    'Google Drive did not mount - re-run this cell and approve the '
    'authorization prompt before continuing.'
)

PROJECT_ROOT = '/content/drive/MyDrive/adaptive-finance-llm'
for sub in ['data', 'checkpoints', 'logs', 'smoke_test']:
    os.makedirs(f'{PROJECT_ROOT}/{sub}', exist_ok=True)

In [ ]:
# Clone (or update, if this cell is re-run) the repo at REPO_BRANCH entirely
# inside Colab's filesystem - nothing from the local Mac is required or
# used. Uses the absolute REPO_PATH (not a path relative to the current
# working directory) so this is safe to re-run even after a previous run of
# this same cell already `%cd`-ed into the repo - a relative check here
# would otherwise look for a nested copy of the repo inside itself and fail.
import os

if os.path.isdir(REPO_PATH):
    %cd {REPO_PATH}
    !git fetch origin {REPO_BRANCH}
    !git checkout {REPO_BRANCH}
    !git pull origin {REPO_BRANCH}
else:
    %cd /content
    !git clone -b {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
    %cd {REPO_PATH}

assert os.getcwd() == REPO_PATH, f'expected cwd {REPO_PATH}, got {os.getcwd()}'

In [ ]:
# Install only what's missing. Deliberately NOT `pip install -r
# requirements.txt`: Colab's preinstalled torch build is already matched to
# this runtime's CUDA version, and requirements.txt's `torch>=2.1` would
# risk pip silently swapping it for a CPU-only or mismatched-CUDA wheel.
# `pytest` is a dev/test-only dependency, not needed to run training or the
# smoke test, so it's left out here too.
!pip install -q numpy pyyaml datasets tokenizers tqdm transformers

In [ ]:
import torch

print('torch version:', torch.__version__)
cuda_ok = torch.cuda.is_available()
print('CUDA available:', cuda_ok)

assert cuda_ok, (
    'CUDA is not available. Check Runtime > Change runtime type > T4 GPU, '
    'and confirm the install cell above did not reinstall a CPU-only torch.'
)

device_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU: {device_name}')
print(f'Total VRAM: {total_vram_gb:.1f} GB')

if 'T4' not in device_name:
    print(
        f"NOTE: expected a Tesla T4 (the project's reference GPU) but got "
        f"'{device_name}' - configs/timings elsewhere assume a T4; a "
        f"different GPU should still work but may behave differently."
    )

In [ ]:
!nvidia-smi

## Smoke test

Builds a tiny model, runs one forward pass, computes loss, runs backward,
then saves and reloads a checkpoint to/from Drive. No dataset download, no
real training — this should finish in a few seconds and print `SMOKE TEST
PASSED`. The checkpoint it writes also confirms Drive write access before
a long training run depends on it.

In [ ]:
!python scripts/smoke_test.py --device cuda --checkpoint-dir "{PROJECT_ROOT}/smoke_test"

## Full training

Only run this after the smoke test above has printed `SMOKE TEST PASSED`.
This downloads the real WikiText-2 dataset (cached to Drive) and runs the
full training config - this is the long-running step, not the smoke test.

In [ ]:
!python scripts/train.py --config configs/baseline_small.yaml

## Resuming

If the runtime disconnects, just remount Drive and re-run the training cell
above — `train.resume: true` in the config means it will pick up from the
latest checkpoint under `checkpoints/` on Drive automatically. See
`SETUP.md` for the exact reproducibility scope of a resumed run.